# NTX Q1 Stage 20 — Post-v3 Frozen Adaptive Semantic Red Team

Creates a new attack suite that is frozen before evaluation. It covers attack families beyond the previously observed matching→all failure and explicitly separates attack generation from detector evaluation.

**Evidence rule:** failed/incomplete third-party infrastructure is recorded as such; no missing score is converted into a model failure or a success.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, time, hashlib, zipfile, random
BUNDLE = Path(os.environ.get("NTX_NOTEBOOK_BUNDLE", ".")).resolve()
if str(BUNDLE) not in sys.path:
    sys.path.insert(0, str(BUNDLE))
from ntx_q1_shared import *
ROOT = ensure_dir(DEFAULT_ROOT)
RESULTS_ROOT = ensure_dir(DEFAULT_RESULTS)
SEED = int(os.environ.get("NTX_SEED", "42"))
random.seed(SEED)
print("ROOT:", ROOT)
print("RESULTS:", RESULTS_ROOT)
print("Frozen repo commit:", REPO_COMMIT)

In [ ]:
import pandas as pd, numpy as np, re, itertools
STAGE="20_adaptive_redteam"
OUT=ensure_dir(RESULTS_ROOT/STAGE)
INP=ensure_dir(ROOT/"inputs"/STAGE)

# Publication run requires a clean seed corpus not used to tune Anchor Lock v3.
seed_path=Path(os.environ.get("NTX_REDTEAM_SEEDS","")) if os.environ.get("NTX_REDTEAM_SEEDS") else None
if seed_path is None or not seed_path.exists():
    pd.DataFrame(columns=["seed_id","raw_text","language","expected_contract_json"]).to_csv(
        OUT/"redteam_seed_REQUIRED_SCHEMA.csv",index=False)
    record_status(OUT,STAGE,"WAITING_FOR_HELDOUT_SEEDS")
    print("Provide NTX_REDTEAM_SEEDS. Do not reuse v3 tuning cases for the publication run.")
else:
    seeds=pd.read_csv(seed_path)
    assert {"seed_id","raw_text","language"}.issubset(seeds.columns)
    print("Held-out seeds:",len(seeds))

In [ ]:
# Attack library is deterministic and frozen by source hash before evaluation.
ATTACKS = {
 "negation_insert": lambda s: "Do not " + s[0].lower()+s[1:] if s else s,
 "negation_remove": lambda s: re.sub(r"\b(do not|don't|never|not)\b\s*","",s,flags=re.I),
 "quantifier_all": lambda s: re.sub(r"\b(matching|selected|these)\b","all",s,flags=re.I),
 "quantifier_subset": lambda s: re.sub(r"\ball\b","matching",s,flags=re.I),
 "only_drop": lambda s: re.sub(r"\bonly\b\s*","",s,flags=re.I),
 "temporal_year_shift": lambda s: re.sub(r"\b(202[0-9])\b",lambda m:str(int(m.group(1))+1),s,count=1),
 "amount_digit_shift": lambda s: re.sub(r"\b(\d{3,})\b",lambda m:str(int(m.group(1))+100),s,count=1),
 "currency_swap": lambda s: re.sub(r"\b(INR|USD|EUR|GBP)\b",lambda m:{"INR":"USD","USD":"INR","EUR":"USD","GBP":"EUR"}[m.group(1)],s,count=1),
 "scope_append_all": lambda s: s.rstrip(".")+" for all records.",
 "exception_clause": lambda s: s.rstrip(".")+", except records marked approved.",
 "double_negation": lambda s: "Do not avoid " + s[0].lower()+s[1:] if s else s,
 "cross_sentence_scope": lambda s: s.rstrip(".")+". Apply this to all matching entities.",
}
attack_source = "\n".join(f"{k}:{getattr(v,'__name__','lambda')}" for k,v in ATTACKS.items())
suite_hash=sha256_text(attack_source)
write_json(OUT/"attack_suite_freeze.json",{"attack_families":list(ATTACKS),"suite_hash":suite_hash,"seed":SEED})
print("Attack suite frozen:",suite_hash)

In [ ]:
if seed_path is not None and seed_path.exists():
    attacks=[]
    for _,r in seeds.iterrows():
        for fam,fn in ATTACKS.items():
            t=fn(str(r.raw_text))
            if t!=str(r.raw_text):
                attacks.append({"seed_id":r.seed_id,"language":r.language,"attack_family":fam,"clean_text":r.raw_text,"attack_text":t})
    atk=pd.DataFrame(attacks).drop_duplicates(["seed_id","attack_family","attack_text"])
    atk.to_json(OUT/"frozen_attack_suite.jsonl",orient="records",lines=True,force_ascii=False)
    write_json(OUT/"attack_suite_counts.json",{
        "n_attacks":len(atk),"n_seeds":int(atk.seed_id.nunique()),
        "families":atk.attack_family.value_counts().to_dict(),
        "suite_sha256":sha256_file(OUT/"frozen_attack_suite.jsonl")
    })
    print("Frozen attacks:",len(atk))

## Evaluation handoff

Run Anchor Lock v3 on the frozen attack JSONL **without changing v3 after seeing these cases**. Save one row per attack to `NTX_REDTEAM_PREDICTIONS` with:

`seed_id, attack_family, detected`

Also evaluate at least 1,000 untouched clean controls and provide `NTX_REDTEAM_CLEAN_PREDICTIONS` with:

`seed_id, triggered`

In [ ]:
pred_path=os.environ.get("NTX_REDTEAM_PREDICTIONS","")
clean_path=os.environ.get("NTX_REDTEAM_CLEAN_PREDICTIONS","")
if pred_path and clean_path and Path(pred_path).exists() and Path(clean_path).exists():
    pr=pd.read_csv(pred_path)
    cl=pd.read_csv(clean_path)
    assert {"seed_id","attack_family","detected"}.issubset(pr.columns)
    assert {"seed_id","triggered"}.issubset(cl.columns)
    pr["detected"]=pr.detected.astype(bool)
    cl["triggered"]=cl.triggered.astype(bool)
    overall=float(pr.detected.mean())
    fpr=float(cl.triggered.mean())
    by=pr.groupby("attack_family").detected.agg(["count","mean"]).reset_index()
    by.to_csv(OUT/"attack_recall_by_family.csv",index=False)
    write_json(OUT/"adaptive_redteam_metrics.json",{
        "n_attacks":len(pr),"attack_recall":overall,"clean_controls":len(cl),"clean_fpr":fpr,
        "suite_frozen_before_eval":True
    })
    paper_ready=(len(pr)>=5000 and len(cl)>=1000)
    record_status(OUT,STAGE,"PASS" if paper_ready else "PARTIAL",paper_ready=paper_ready,
                  attack_recall=overall,clean_fpr=fpr,n_attacks=len(pr),n_clean=len(cl))
    display(by)
else:
    print("Predictions not supplied yet. Frozen attack suite is ready for evaluation.")

In [ ]:
z=package_stage(OUT,"NTX_Q1_20_ADAPTIVE_REDTEAM_RESULTS.zip")
print("Created:",z)
maybe_colab_download(z)